In [3]:
import os
import gymnasium as gym
import pandas as pd
from skopt.space import Real
from multiprocessing import Pool
from tqdm import tqdm


from popy.simulation_tools import *
from popy.io_tools import load_behavior, load_behavior_yuri
from popy.behavior_data_tools import *
from popy.plotting.plotting_tools import *
from popy.config import PROJECT_PATH_LOCAL

from popy.simulation_helpers import fit_simulate, simulate_agent
from popy.plotting.plotting_tools import show_target_selection, show_target_selection_compact

MODELS = {
    "Repeating agent": {
        "agent_class": RepeatingAgent,
        "fixed_params": {},
        "free_params": ["epsilon"],
    },

    "WSLS": {
        "agent_class": WSLSAgent,
        "fixed_params": {},
        "free_params": ["epsilon"],
    },

    "Standard RL": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": False},
        "free_params": ["alpha", "beta"],
    },
    "Standard RL - stickiness": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": False},
        "free_params": ["alpha", "beta", "stickiness_bias"],
    },
    "Standard RL - forgetting": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": False},
        "free_params": ["alpha", "beta", "forgetting_rate", "forgetting_threshold"],
    },
    "Standard RL - stickiness + forgetting": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": False},
        "free_params": ["alpha", "beta", "forgetting_rate", "forgetting_threshold", "stickiness_bias"],
    },

    "Inferential RL": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": True},
        "free_params": ["alpha", "beta"],
    },
    "Inferential RL - stickiness": {    
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": True},
        "free_params": ["alpha", "beta", "stickiness_bias"],
    },
    "Inferential RL - stickiness + spatial bias": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": True},
        "free_params": ["alpha", "beta", "stickiness_bias", "b2_bias"],
    },
    "Inferential RL - stickiness + multiple alphas": {
        "agent_class": QLearner,
        "fixed_params": {"structure_aware": True},
        "free_params": ["alpha", "alpha_unchosen", "beta", "stickiness_bias"],
    },

    "Foraging - no reset": {
        "agent_class": ForagingAgent,
        "fixed_params": {"reset_on_switch": False},
        "free_params": ["alpha", "beta", "V0"],
    },
    "Foraging": {
        "agent_class": ForagingAgent,
        "fixed_params": {"reset_on_switch": True},
        "free_params": ["alpha", "beta", "V0"],
    },
    "Foraging - abandoned bias": {
        "agent_class": ForagingAgent,
        "fixed_params": {"reset_on_switch": True},
        "free_params": ["alpha", "beta",  "V0", "abandoned_bias", "abandoned_decay"],
    },
    "Foraging - abandoned bias + spatial bias": {
        "agent_class": ForagingAgent,
        "fixed_params": {"reset_on_switch": True},
        "free_params": ["alpha", "beta",  "V0", "abandoned_bias", "abandoned_decay", "b2_bias"],
    },
    "Foraging - adaptive threshold": {
        "agent_class": ForagingAgentAdaptive,
        "fixed_params": {'alpha_threshold': 0.05},
        "free_params": [],
    },
    "POMDPAgent": {
        "agent_class": POMDPAgent,
        "fixed_params": {},
    },

}

In [ ]:
model_name = 'POMDPAgent'

env = gym.make(
    "zsombi/monkey-bandit-task-v0", 
    n_arms=3, 
    max_episode_steps=100_000,
)

"""Optimize a single model and return results."""
model_params = MODELS[model_name]

agent_class = model_params["agent_class"]
fixed_params = model_params["fixed_params"]
params = {}

# Simulate behavior with best parameters
behav = simulate_agent(
    agent_class=agent_class, 
    params=params,
    env=env, 
    fixed_params=fixed_params, 
    behavioral_variables=[],
)
reward_rate = behav['reward'].mean()
proba_best = (behav['action'] == behav['best_arm']).mean()

print(f"Simulated behavior with model: {model_name}")
print(f"Reward rate: {reward_rate:.3f}")
print(f"Proba. choosing best target: {proba_best:.3f}")

In [ ]:
behav['monkey'] = ''
behav['session'] = '0'
behav = convert_column_format(behav)

behav = behav[:100]

# plot behavior 
# target selection
#savedir = os.path.join('figs', 'example_session_compact.svg')
savedir = None
show_target_selection_compact(behav, 
                              format='poster',
                              background_values=['V', 'V0'],   # here we tell what variable to use as background - value in this case
                              #background_values=['Q_1', 'Q_2', 'Q_3'],   # here we tell what variable to use as background - value in this case
                              title=None, #f'monkey: {monkey}, session: {session}', 
                              show=True, 
                              savedir=savedir)

In [ ]:
# keep only models in agents_to_show.keys() and renaming them
behav['model'] = model_name

# Apply the function to create the monkey column
behav['monkey'] = behav['model']
behav['session'] = 0

behav = convert_column_format(behav, original='simulation')

behav = add_history_of_feedback(behav, num_trials=3, one_column=False, coding=(0, 1))
behav = add_switch_info(behav, add_trials_since_switch=True, flip_coding=False)
behav = behav.dropna()

fig1, _ = plot_hist_thingy(behav, paper_format=False)
# remove legend
plt.legend([], [], frameon=False)
fig2, _ = plot_strategy(behav, paper_format=False, h=2, w=1.6, ylim=[0, .4], verbose=True)
# remove legend
plt.legend([], [], frameon=False)


KeyError: 'action'